# Iris Dataset Classification using CRISP-DM

## 1. Business Understanding

The goal is to accurately classify Iris flowers into one of three species (Setosa, Versicolor, Virginica) based on their sepal and petal measurements. This classification can be useful for botanical studies, automated plant identification systems, or educational purposes. The key metric for success is high classification accuracy.

## 2. Data Understanding

### Load the Dataset and Display Head

In [79]:
import pandas as pd
from sklearn.datasets import load_iris

# Load the Iris dataset
iris = load_iris(as_frame=True)

# Convert to DataFrame
df = (
    iris.frame
    if iris.frame is not None
    else pd.DataFrame(iris.data, columns=iris.feature_names)
)
df["target"] = iris.target

# Display the first few rows
print("--- Dataset Head ---")
df.head()

--- Dataset Head ---


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


### Display Dataset Information

In [80]:
# Display information about the dataset (data types, non-null counts)
print("--- Dataset Info ---")
df.info()

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


### Display Target Distribution

In [81]:
# Display the distribution of the target variable
print("--- Target Distribution ---")
print(df["target"].value_counts())

--- Target Distribution ---
target
0    50
1    50
2    50
Name: count, dtype: int64


### Display Dataset Description

In [82]:
# Display basic statistics for numerical features
print("--- Dataset Description ---")
df.describe()

--- Dataset Description ---


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


## 3. Data Preparation

In [83]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = iris.data
y = iris.target

# Split data into training and testing sets (70% train, 30% test)
# Use stratify to ensure the target distribution is similar in train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Initialize StandardScaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Data Preparation Complete ---")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
print(
    f"X_train_scaled shape: {X_train_scaled.shape}, X_test_scaled shape: {X_test_scaled.shape}"
)

--- Data Preparation Complete ---
X_train shape: (105, 4), y_train shape: (105,)
X_test shape: (45, 4), y_test shape: (45,)
X_train_scaled shape: (105, 4), X_test_scaled shape: (45, 4)


## 4. Modeling

In [84]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

### Model 1: Logistic Regression

In [85]:
# Define hyperparameter grid for Logistic Regression
param_grid_lr = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["liblinear"],
}

# Initialize GridSearchCV for Logistic Regression
grid_search_lr = GridSearchCV(
    LogisticRegression(random_state=42), param_grid_lr, cv=5, scoring="accuracy"
)

# Fit the grid search to the training data
grid_search_lr.fit(X_train_scaled, y_train)

print("--- Logistic Regression Model Training Complete ---")
print("Best parameters for Logistic Regression:", grid_search_lr.best_params_)
print(
    "Best cross-validation accuracy score for Logistic Regression:",
    grid_search_lr.best_score_,
)

# Store the best model
best_lr = grid_search_lr.best_estimator_

--- Logistic Regression Model Training Complete ---
Best parameters for Logistic Regression: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-validation accuracy score for Logistic Regression: 0.9714285714285713


### Model 2: Random Forest

In [86]:
# Define hyperparameter grid for Random Forest
param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"],
}

# Initialize GridSearchCV for Random Forest
grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring="accuracy"
)

# Fit the grid search to the training data
grid_search_rf.fit(X_train, y_train)

print("--- Random Forest Model Training Complete ---")
print("Best parameters for Random Forest:", grid_search_rf.best_params_)
print(
    "Best cross-validation accuracy score for Random Forest:",
    grid_search_rf.best_score_,
)

# Store the best model
best_rf = grid_search_rf.best_estimator_

--- Random Forest Model Training Complete ---
Best parameters for Random Forest: {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}
Best cross-validation accuracy score for Random Forest: 0.9619047619047618


### Model 3: XGBoost

In [87]:
# Define hyperparameter grid for XGBoost
param_grid_xgb = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.3],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
}

# Initialize GridSearchCV for XGBoost
grid_search_xgb = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric="logloss"),
    param_grid_xgb,
    cv=5,
    scoring="accuracy",
)

# Fit the grid search to the training data
grid_search_xgb.fit(X_train_scaled, y_train)

print("--- XGBoost Model Training Complete ---")
print("Best parameters for XGBoost:", grid_search_xgb.best_params_)
print("Best cross-validation accuracy score for XGBoost:", grid_search_xgb.best_score_)

# Store the best model
best_xgb = grid_search_xgb.best_estimator_

--- XGBoost Model Training Complete ---
Best parameters for XGBoost: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}
Best cross-validation accuracy score for XGBoost: 0.9523809523809523


## 5. Evaluation

In [88]:
from sklearn.metrics import classification_report, accuracy_score

### Evaluate Logistic Regression

In [89]:
# Evaluate Logistic Regression on test set
print("--- Logistic Regression Performance on Test Set ---")
y_pred_lr = best_lr.predict(X_test_scaled)
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {accuracy_lr:.4f}")

# Use target_names for clear output: 'setosa' (0), 'versicolor' (1), 'virginica' (2)
print(classification_report(y_test, y_pred_lr, target_names=iris.target_names))

--- Logistic Regression Performance on Test Set ---
Accuracy: 0.8667
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.80      0.80      0.80        15
   virginica       0.80      0.80      0.80        15

    accuracy                           0.87        45
   macro avg       0.87      0.87      0.87        45
weighted avg       0.87      0.87      0.87        45



### Evaluate Random Forest

In [90]:
# Evaluate Random Forest on test set
print("--- Random Forest Performance on Test Set ---")
y_pred_rf = best_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Accuracy: {accuracy_rf:.4f}")

# Use target_names for clear output: 'setosa' (0), 'versicolor' (1), 'virginica' (2)
print(classification_report(y_test, y_pred_rf, target_names=iris.target_names))

--- Random Forest Performance on Test Set ---
Accuracy: 0.9111
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.82      0.93      0.88        15
   virginica       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45



In [91]:
# Evaluate XGBoost on test set
print("--- XGBoost Performance on Test Set ---")
y_pred_xgb = best_xgb.predict(X_test_scaled)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy: {accuracy_xgb:.4f}")

# Use target_names for clear output: 'setosa' (0), 'versicolor' (1), 'virginica' (2)
print(classification_report(y_test, y_pred_xgb, target_names=iris.target_names))

--- XGBoost Performance on Test Set ---
Accuracy: 0.9111
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.82      0.93      0.88        15
   virginica       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45



## Model Comparison

In [92]:
model_performance = {
    "Logistic Regression": accuracy_lr,
    "Random Forest": accuracy_rf,
    "XGBoost": accuracy_xgb,
}

print("--- Model Performance Comparison (Accuracy on Test Set) ---")
for model_name, accuracy in model_performance.items():
    print(f"{model_name}: {accuracy:.4f}")

# Determine the best performing model based on overall accuracy
best_model_name = max(model_performance, key=model_performance.get)
best_test_accuracy = model_performance[best_model_name]

print(
    f"\n--- Best Performing Model (based on Accuracy): {best_model_name} (Accuracy: {best_test_accuracy:.4f}) ---"
)

# Select the actual best model object based on the comparison
if best_model_name == "Logistic Regression":
    final_best_model = best_lr
    requires_scaling = True
elif best_model_name == "Random Forest":
    final_best_model = best_rf
    requires_scaling = False
elif best_model_name == "XGBoost":
    final_best_model = best_xgb
    requires_scaling = True
else:
    # Fallback - should not happen if best_model_name is one of the keys
    final_best_model = best_xgb
    requires_scaling = True

--- Model Performance Comparison (Accuracy on Test Set) ---
Logistic Regression: 0.8667
Random Forest: 0.9111
XGBoost: 0.9111

--- Best Performing Model (based on Accuracy): Random Forest (Accuracy: 0.9111) ---


## 6. Deployment

In [93]:
import os

# Define a base directory for saving models for different projects/datasets
base_model_save_dir = os.path.join(os.pardir, "saved_models")

# Define a specific directory for this project's models
project_model_dir = os.path.join(base_model_save_dir, "iris_classification")

# Create the directory if it doesn't exist
os.makedirs(project_model_dir, exist_ok=True)
print(f"Ensured directory exists: {project_model_dir}")


# Define the filenames for the model and scaler within the project directory
model_filename = os.path.join(
    project_model_dir, f"{best_model_name.replace(' ', '_').lower()}_model.joblib"
)
scaler_filename = os.path.join(project_model_dir, "scaler.joblib")

# Save the best performing model
joblib.dump(final_best_model, model_filename)
print(f"Best model ({best_model_name}) saved to: {model_filename}")

# Save the scaler
joblib.dump(scaler, scaler_filename)
print(f"Scaler saved to: {scaler_filename}")


# Save best model configuration
config_filename = os.path.join(project_model_dir, "model_config.txt")
with open(config_filename, "w") as f:
    f.write(f"Best_Model: {best_model_name}\n")
    f.write(f"Requires_Scaling: {requires_scaling}\n")
    f.write(f"Target_Names: {iris.target_names.tolist()}\n")
print(f"Model configuration saved to: {config_filename}")

Ensured directory exists: ..\saved_models\iris_classification
Best model (Random Forest) saved to: ..\saved_models\iris_classification\random_forest_model.joblib
Scaler saved to: ..\saved_models\iris_classification\scaler.joblib
Model configuration saved to: ..\saved_models\iris_classification\model_config.txt
